In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === Load data ===
df = pd.read_csv(
    r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Forest_Area_Change_historical_Disturbance_Attribution_1988_2021_pa&ownership_reprojected.csv"
)

# === Map states to regions ===
state_to_region_division = {
    'Connecticut': ('Northeast', 'New England'), 'Maine': ('Northeast', 'New England'),
    'Massachusetts': ('Northeast', 'New England'), 'New Hampshire': ('Northeast', 'New England'),
    'Rhode Island': ('Northeast', 'New England'), 'Vermont': ('Northeast', 'New England'),
    'New Jersey': ('Northeast', 'Middle Atlantic'), 'New York': ('Northeast', 'Middle Atlantic'),
    'Pennsylvania': ('Northeast', 'Middle Atlantic'),
    'Illinois': ('Midwest', 'East North Central'), 'Indiana': ('Midwest', 'East North Central'),
    'Michigan': ('Midwest', 'East North Central'), 'Ohio': ('Midwest', 'East North Central'),
    'Wisconsin': ('Midwest', 'East North Central'),
    'Iowa': ('Midwest', 'West North Central'), 'Kansas': ('Midwest', 'West North Central'),
    'Minnesota': ('Midwest', 'West North Central'), 'Missouri': ('Midwest', 'West North Central'),
    'Nebraska': ('Midwest', 'West North Central'), 'North Dakota': ('Midwest', 'West North Central'),
    'South Dakota': ('Midwest', 'West North Central'),
    'Delaware': ('South', 'South Atlantic'), 'District of Columbia': ('South', 'South Atlantic'),
    'Florida': ('South', 'South Atlantic'), 'Georgia': ('South', 'South Atlantic'),
    'Maryland': ('South', 'South Atlantic'), 'North Carolina': ('South', 'South Atlantic'),
    'South Carolina': ('South', 'South Atlantic'), 'Virginia': ('South', 'South Atlantic'),
    'West Virginia': ('South', 'South Atlantic'),
    'Alabama': ('South', 'East South Central'), 'Kentucky': ('South', 'East South Central'),
    'Mississippi': ('South', 'East South Central'), 'Tennessee': ('South', 'East South Central'),
    'Arkansas': ('South', 'West South Central'), 'Louisiana': ('South', 'West South Central'),
    'Oklahoma': ('South', 'West South Central'), 'Texas': ('South', 'West South Central'),
    'Arizona': ('West', 'Mountain'), 'Colorado': ('West', 'Mountain'), 'Idaho': ('West', 'Mountain'),
    'Montana': ('West', 'Mountain'), 'Nevada': ('West', 'Mountain'), 'New Mexico': ('West', 'Mountain'),
    'Utah': ('West', 'Mountain'), 'Wyoming': ('West', 'Mountain'),
    'Alaska': ('West', 'Pacific'), 'California': ('West', 'Pacific'),
    'Hawaii': ('West', 'Pacific'), 'Oregon': ('West', 'Pacific'), 'Washington': ('West', 'Pacific')
}
df['Region'] = df['Ecoregion'].map(lambda x: state_to_region_division.get(x, ('Unknown',))[0])

# === Classify gain / loss / stable ===
df['ChangeType'] = df['ForestChangeType'].apply(
    lambda x: 'Forest Gain' if x in [1, 2, 3, 4, 5]
    else 'Forest Loss' if x in [10, 20, 30, 40, 50]
    else 'Stable Forest'
)

df['Area_km2'] = df['PixelCount'] * 0.0009
df['Year'] = df['Year_To']

# === Disturbance mapping and colors ===
disturbance_rename = {
    'No Disturbance': 'No Disturbance Detected',
    'Forest Management': 'Logging',
    'Construction': 'Construction',
    'Stress': 'Stress',
    'Natural Hazard': 'Natural Hazard',
    'Water Dynamic': 'Water Dynamic',
    'Fire': 'Fire',
    'Agriculture Activity': 'Agriculture Activity',
    'Other': 'Other'
}
disturbance_colors = {
    'No Disturbance Detected': 'black',
    'Logging': '#1b9e77',
    'Construction': 'purple',
    'Stress': '#e7298a',
    'Natural Hazard': '#66a61e',
    'Water Dynamic': '#1f78b4',
    'Fire': 'red',
    'Agriculture Activity': 'gold',
    'Other': 'gray'
}
# === Final (merged) display labels & colors ===
merged_labels = [
    'Logging', 'Construction', 'Stress', 'Natural Hazard',
    'Water Dynamic', 'Fire', 'Agriculture Activity', 'Others'
]
merged_colors = {
    'Logging': '#1b9e77',
    'Construction': 'purple',
    'Stress': '#e7298a',
    'Natural Hazard': '#66a61e',
    'Water Dynamic': '#1f78b4',
    'Fire': 'red',
    'Agriculture Activity': 'gold',
    'Others': 'gray'
}

raw_disturbances = list(disturbance_rename.keys())
display_disturbances = list(disturbance_rename.values())

# === Step 0: Read precomputed forest area from CSV ===
csv_path = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\forest_area_by_region.csv"

# Load the CSV
forest_area_df = pd.read_csv(csv_path)

# Pivot the table to have years as index and regions as columns
region_forest_area = forest_area_df.pivot_table(
    index='Year',
    columns='Region',
    values='ForestArea_km2',
    aggfunc='sum',
    fill_value=0
)

# (Optional) sort index and columns for consistency
region_forest_area = region_forest_area.sort_index().sort_index(axis=1)

# === Step 1: Compute regional net disturbance DataFrames ===
region_net_change = {}
region_forest_area_km2 = {}

for region in ['Northeast', 'Midwest', 'South', 'West']:
    region_df = df[df['Region'] == region]

    gain = region_df[region_df['ChangeType'] == 'Forest Gain']
    loss = region_df[region_df['ChangeType'] == 'Forest Loss']

    gain_grp = gain.groupby(['Year', 'DisturbanceCategory'])['Area_km2'].sum().unstack().fillna(0)
    loss_grp = loss.groupby(['Year', 'DisturbanceCategory'])['Area_km2'].sum().unstack().fillna(0)

    net_disturbance = gain_grp - loss_grp

    # Ensure all disturbance types are present
    for d in raw_disturbances:
        if d not in net_disturbance.columns:
            net_disturbance[d] = 0

    net_disturbance = net_disturbance[raw_disturbances]
    net_disturbance = net_disturbance.rename(columns=disturbance_rename)
    net_disturbance = net_disturbance.loc[1989:]  # remove early years

    # --- MERGE "No Disturbance Detected" + "Other" -> "Others" ---
    net_disturbance['Others'] = net_disturbance.get('Other', 0) + net_disturbance.get('No Disturbance Detected', 0)
    net_disturbance = net_disturbance.drop(columns=[c for c in ['Other', 'No Disturbance Detected'] if c in net_disturbance.columns])

    # Ensure consistent column order for plotting
    for c in merged_labels:
        if c not in net_disturbance.columns:
            net_disturbance[c] = 0
    net_disturbance = net_disturbance[merged_labels]

    region_net_change[region] = net_disturbance

    forest_area = region_forest_area[region].reindex(net_disturbance.index, fill_value=0)
    region_forest_area_km2[region] = forest_area

In [ ]:
# === Step 2: Plot ===
fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex=True, dpi=600)
axes = axes.flatten()

for idx, region in enumerate(['Northeast', 'Midwest', 'South', 'West']):
    ax = axes[idx]
    net_disturbance = region_net_change[region]

    pos = net_disturbance.clip(lower=0)
    neg = net_disturbance.clip(upper=0)
    bar_x = range(len(net_disturbance))

    pos_colors = [merged_colors[d] for d in pos.columns]
    neg_colors = [merged_colors[d] for d in neg.columns]

    pos.plot(kind='bar', stacked=True, ax=ax, color=pos_colors, width=0.9, legend=False)
    neg.plot(kind='bar', stacked=True, ax=ax, color=neg_colors, width=0.9, legend=False)

    ax.axhline(0, color='gray', linestyle=':')
    ax.set_title(f"({chr(97+idx)}) {region}", fontsize=12, fontweight='bold', loc='left')
    ax.set_ylabel("Net Forest Area Change (km²)")
    ax.set_xticks(bar_x)
    ax.set_xticklabels(net_disturbance.index, rotation=45)

    # Forest area line (still national-level)
    ax_right = ax.twinx()
    forest_area = region_forest_area[region].reindex(net_disturbance.index, fill_value=0)
    ax_right.plot(bar_x, forest_area.values, color='green', linewidth=2, linestyle='-', marker='x')
    ax_right.set_ylabel("Total Forest Area (km²)", color='green')
    ax_right.tick_params(axis='y', colors='green')

    ax.ticklabel_format(axis='y', style='sci', scilimits=(3, 3))
    ax_right.ticklabel_format(axis='y', style='sci', scilimits=(6, 6))

# Legend
handles = [plt.Rectangle((0, 0), 1, 1, color=merged_colors[k]) for k in merged_labels]
axes[-1].legend(handles, merged_labels, bbox_to_anchor=(0.05, 0.5), loc='upper left')

plt.tight_layout()
plt.savefig(r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\key_outputs\Figure_S3.png", dpi=600, bbox_inches='tight')
plt.show()